# AquaVision — Huấn luyện mô hình phát hiện cá bằng YOLO (Ultralytics)

Notebook này huấn luyện một mô hình **object detection** họ YOLO (YOLOv8 → YOLO26) trên Google Colab,
sử dụng dữ liệu ảnh cá (Koi / Tilapia / Eel) được quản lý và gán nhãn trong dự án AquaVision, tải trực
tiếp từ Roboflow.

**Quy trình:**

1. Kiểm tra GPU và runtime
2. Gắn Google Drive để lưu trữ dữ liệu & kết quả lâu dài
3. Cài đặt thư viện
4. Cấu hình huấn luyện — **một cell duy nhất**, chỉnh sửa ở đây, các cell phía dưới không hard-code lại
5. Xác thực & tải dataset từ Roboflow, kiểm tra tính toàn vẹn
6. Thiết lập reproducibility (seed)
7. Huấn luyện mô hình (tự resume nếu Colab bị ngắt phiên giữa chừng)
8. Đánh giá mô hình trên tập test
9. Trực quan hoá kết quả (PR curve, confusion matrix, sample predictions)
10. Suy luận thử nghiệm nhanh
11. Export mô hình (ONNX / TensorRT — tuỳ chọn)
12. Ghi manifest (thông số + kết quả) về Google Drive để truy vết
13. Tổng kết

> **Trước khi chạy:** vào `Runtime > Change runtime type` và chọn GPU (T4 trở lên được khuyến nghị).
>
> **Cần chuẩn bị:** một Roboflow API key. Khuyến nghị lưu vào Colab Secrets (biểu tượng 🔑 ở thanh bên
> trái) với tên `ROBOFLOW_API_KEY` thay vì dán trực tiếp vào notebook.

## 1. Kiểm tra môi trường GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "Không tìm thấy GPU. Vào Runtime > Change runtime type > Hardware accelerator "
    "và chọn GPU (khuyến nghị T4 trở lên) trước khi tiếp tục."
)

print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"CUDA    : {torch.version.cuda}")
print(f"PyTorch : {torch.__version__}")

## 2. Gắn Google Drive

Dataset, checkpoint và toàn bộ kết quả huấn luyện được ghi trực tiếp vào Google Drive để:

- Không mất dữ liệu khi Colab tự ngắt phiên (idle timeout / disconnect)
- Có thể **resume** huấn luyện từ checkpoint gần nhất mà không cần tải lại dataset hoặc bắt đầu lại từ đầu

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Cài đặt thư viện

In [ ]:
%pip install -q "ultralytics>=8.3.0" "roboflow>=1.1.0"

## 4. Cấu hình huấn luyện

Đây là **cell cấu hình duy nhất** của notebook. Điều chỉnh dataset Roboflow, kiến trúc model và
hyperparameter tại đây.

- `model.version` hỗ trợ: `YOLO26`, `YOLO12`, `YOLO11`, `YOLOv10`, `YOLOv9`, `YOLOv8`
- `model.size` hỗ trợ: `n`, `s`, `m`, `l`, `x`
- `roboflow.format`: định dạng export tương thích Ultralytics — `yolov8`/`yolov11`/`yolo26`... đều dùng
  chung cấu trúc nhãn `.txt` + `data.yaml`, có thể dùng lẫn cho mọi version YOLO ở trên

In [ ]:
import json

CONFIG = {
    "project": {
        # Dùng để đặt tên thư mục run trên Drive — nên gồm loài cá + kiến trúc model
        "name": "aquavision-koi-yolo",
    },
    "roboflow": {
        "workspace": "dd-1pubd",
        "project": "koi-f2dp4",
        "version": 2,
        "format": "yolov11",
    },
    "model": {
        "version": "YOLO26",  # YOLO26 | YOLO12 | YOLO11 | YOLOv10 | YOLOv9 | YOLOv8
        "size": "m",           # n | s | m | l | x
    },
    "hyperparameters": {
        "epochs": 150,
        "batch_size": 16,
        "image_size": 640,
        "workers": 8,
        "patience": 30,        # early stopping — số epoch không cải thiện trước khi dừng
        "optimizer": "auto",
        "amp": True,           # mixed precision (fp16)
        "device": 0,           # 0 = GPU đầu tiên, "cpu" nếu chạy trên CPU
        "seed": 42,
        # Đặt True nếu phiên Colab trước đó bị ngắt giữa chừng — notebook sẽ tự
        # tiếp tục từ checkpoint "last.pt" gần nhất thay vì huấn luyện lại từ đầu.
        "resume": False,
    },
    "export": {
        "onnx": False,
        "tensorrt": False,
    },
    "drive": {
        # Toàn bộ dataset/checkpoint/kết quả được ghi vào đây để không mất khi mất phiên Colab
        "root": "/content/drive/MyDrive/AquaVision",
    },
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

## 5. Xác thực & tải dataset từ Roboflow

API key được lấy an toàn qua Colab Secrets dưới tên `ROBOFLOW_API_KEY`. Nếu chưa cấu hình secret,
notebook sẽ yêu cầu nhập trực tiếp (giá trị chỉ tồn tại trong runtime hiện tại, không được lưu vào file).

In [ ]:
import os
from getpass import getpass

ROBOFLOW_API_KEY = None

try:
    from google.colab import userdata

    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass("Nhập Roboflow API key (không được lưu lại trong notebook): ")

os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

In [ ]:
from pathlib import Path

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
rf_project = rf.workspace(CONFIG["roboflow"]["workspace"]).project(CONFIG["roboflow"]["project"])
rf_version = rf_project.version(CONFIG["roboflow"]["version"])

dataset_dir = Path("/content/datasets") / CONFIG["project"]["name"]
dataset = rf_version.download(CONFIG["roboflow"]["format"], location=str(dataset_dir))

print(f"Dataset đã tải về: {dataset.location}")

In [ ]:
import yaml

data_yaml_path = Path(dataset.location) / "data.yaml"
assert data_yaml_path.exists(), f"Không tìm thấy data.yaml tại {data_yaml_path}"

with open(data_yaml_path) as f:
    data_yaml = yaml.safe_load(f)

print(f"Số lớp (nc) : {data_yaml['nc']}")
print(f"Tên lớp     : {data_yaml['names']}")

for split in ("train", "valid", "test"):
    split_images_dir = Path(dataset.location) / split / "images"
    if split_images_dir.exists():
        count = len(list(split_images_dir.glob("*")))
        print(f"{split:>6}: {count} ảnh")
    else:
        print(f"{split:>6}: (không có thư mục — bỏ qua)")

## 6. Thiết lập reproducibility

Cố định seed cho toàn bộ nguồn ngẫu nhiên để kết quả có thể tái lập giữa các lần chạy.

In [ ]:
import random

import numpy as np


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["hyperparameters"]["seed"])

## 7. Huấn luyện mô hình

Checkpoint pretrained được resolve theo `model.version` + `model.size`. Kết quả (`weights/best.pt`,
`weights/last.pt`, biểu đồ) được ghi trực tiếp vào Google Drive (`project=`, `name=`).

Nếu phiên Colab bị ngắt giữa chừng: đặt `CONFIG["hyperparameters"]["resume"] = True` ở Bước 4 rồi chạy
lại notebook từ đầu — Ultralytics sẽ tự đọc `last.pt` và tiếp tục đúng từ epoch bị ngắt.

In [ ]:
_YOLO_VERSION_PREFIXES = {
    "YOLO26": "yolo26",
    "YOLO12": "yolo12",
    "YOLO11": "yolo11",
    "YOLOv10": "yolov10",
    "YOLOv9": "yolov9",
    "YOLOv8": "yolov8",
}


def resolve_yolo_checkpoint(version: str, size: str) -> str:
    prefix = _YOLO_VERSION_PREFIXES.get(version)
    if prefix is None:
        supported = ", ".join(_YOLO_VERSION_PREFIXES)
        raise ValueError(f"Version '{version}' không được hỗ trợ (hỗ trợ: {supported})")
    return f"{prefix}{size}.pt"

In [ ]:
from ultralytics import YOLO

model_cfg = CONFIG["model"]
hyperparams = CONFIG["hyperparameters"]

checkpoint = resolve_yolo_checkpoint(model_cfg["version"], model_cfg["size"])
print(f"Checkpoint pretrained: {checkpoint}")

runs_dir = Path(CONFIG["drive"]["root"]) / "runs" / "yolo"
run_name = CONFIG["project"]["name"]
last_checkpoint = runs_dir / run_name / "weights" / "last.pt"

if hyperparams["resume"] and last_checkpoint.exists():
    print(f"Tiếp tục huấn luyện từ checkpoint: {last_checkpoint}")
    model = YOLO(str(last_checkpoint))
    results = model.train(resume=True)
else:
    if hyperparams["resume"]:
        print("resume=True nhưng không tìm thấy checkpoint trước đó — huấn luyện mới từ pretrained.")
    model = YOLO(checkpoint)
    results = model.train(
        data=str(data_yaml_path),
        epochs=hyperparams["epochs"],
        batch=hyperparams["batch_size"],
        imgsz=hyperparams["image_size"],
        workers=hyperparams["workers"],
        patience=hyperparams["patience"],
        optimizer=hyperparams["optimizer"],
        amp=hyperparams["amp"],
        device=hyperparams["device"],
        seed=hyperparams["seed"],
        project=str(runs_dir),
        name=run_name,
        exist_ok=True,
    )

save_dir = Path(results.save_dir)
best_weights = save_dir / "weights" / "best.pt"
last_weights = save_dir / "weights" / "last.pt"

print(f"Best weights: {best_weights}")
print(f"Last weights: {last_weights}")

## 8. Đánh giá mô hình

In [ ]:
best_model = YOLO(best_weights)

metrics = best_model.val(data=str(data_yaml_path), imgsz=hyperparams["image_size"], split="test")

print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

## 9. Trực quan hoá kết quả

In [ ]:
from IPython.display import Image, display

for name in ("results.png", "confusion_matrix.png", "BoxPR_curve.png", "val_batch0_pred.jpg"):
    path = save_dir / name
    if path.exists():
        display(Image(filename=str(path), width=800))
    else:
        print(f"(bỏ qua — không tìm thấy {name})")

## 10. Suy luận thử nghiệm

In [ ]:
test_images_dir = Path(dataset.location) / "test" / "images"

if test_images_dir.exists():
    sample_images = sorted(test_images_dir.glob("*"))[:5]
    predict_results = best_model.predict(
        source=[str(p) for p in sample_images],
        conf=0.25,
        imgsz=hyperparams["image_size"],
        save=True,
        project=str(runs_dir),
        name=f"{run_name}-predict",
        exist_ok=True,
    )
    for r in predict_results:
        pred_path = Path(r.save_dir) / Path(r.path).name
        display(Image(filename=str(pred_path), width=600))
else:
    print("Không có thư mục test/images — bỏ qua suy luận thử nghiệm.")

## 11. Export mô hình (tuỳ chọn)

Bật `CONFIG["export"]["onnx"]` / `["tensorrt"]` ở Bước 4 nếu cần triển khai lên edge device
(khớp với cấu hình `export` trong `SlothStudio/config/slothstudio.yml`).

In [ ]:
export_cfg = CONFIG["export"]

if export_cfg["onnx"]:
    onnx_path = best_model.export(format="onnx", imgsz=hyperparams["image_size"], dynamic=True, simplify=True)
    print(f"ONNX export: {onnx_path}")

if export_cfg["tensorrt"]:
    engine_path = best_model.export(format="engine", imgsz=hyperparams["image_size"], half=True)
    print(f"TensorRT export: {engine_path}")

## 12. Ghi manifest kết quả

Lưu lại cấu hình + metrics của lần chạy này để truy vết sau này (dataset version, hyperparameter,
checkpoint dùng, kết quả đạt được).

In [ ]:
import json
from datetime import datetime

run_manifest = {
    "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "config": CONFIG,
    "checkpoint": checkpoint,
    "best_weights": str(best_weights),
    "last_weights": str(last_weights),
    "metrics": {
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    },
}

manifest_path = save_dir / "run_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(run_manifest, f, indent=2, default=str, ensure_ascii=False)

print(f"Manifest: {manifest_path}")

## 13. Tổng kết

In [ ]:
print("Huấn luyện hoàn tất")
print(f"  Model         : {checkpoint}")
print(f"  Best weights  : {best_weights}")
print(f"  mAP50-95      : {metrics.box.map:.4f}")
print(f"  Kết quả lưu tại: {save_dir}")